In [ ]:

# required imports:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import precision_score, recall_score
import pandas as pd
from sklearn.metrics import classification_report
import emoji
from transformers import pipeline

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:

def load_data():

    all_files = []

    for file in glob.glob("data/*.csv"):
        df = pd.read_csv(file, on_bad_lines="skip", engine="python")
        df = df.dropna(subset=["Text"])
        lbl = file.split("/")[-1].replace(".csv", "")
        df["label"] = lbl

        all_files.append(df[["Text", "label"]])
    return pd.concat(all_files, ignore_index=True)


In [ ]:

def main(repeat=True):

    df = load_data()

    if repeat:
        print("Number of classes:", df["label"].nunique())
        print("Total number of samples in dataframe:", len(df))


    X = df["Text"].astype(str)
    y = df["label"].astype(str)

    # applying train test split
    X_train, X_values, y_train, y_values = train_test_split(X,y,test_size=0.3,random_state=40,stratify=y )


    vectorize = TfidfVectorizer()
    X_train = vectorize.fit_transform(X_train)
    X_values2 = vectorize.transform(X_values)


    model = LinearSVC()

    model.fit(X_train, y_train)
    predictions = model.predict(X_values2)
    accuracy = accuracy_score(y_values, predictions)
    # average= "macro" because there are many classes where emojis may vary in balance.
    f1_sc = f1_score(y_values, predictions, average= "macro")


    print("Classification by emoji:\n")
    print(classification_report(y_values, predictions))
    print("\nSVM Accuracy:", accuracy)
    print("\nSVM Macro f1 Score", f1_sc)



    labels = sorted(df["label"].unique())

    prec = precision_score(y_values, predictions, average=None, labels=labels)
    rec = recall_score(y_values, predictions, average=None, labels=labels)


    print("labels:", labels[:10] , "and more...")
    print("Precisions:", prec[:10] , "and more...")
    print("Recalls:", rec[:10] , "and more...")
    return df, y_values, predictions





main()

Number of classes: 43
Total number of samples in dataframe: 860028
Classification by emoji:

                               precision    recall  f1-score   support

backhand_index_pointing_right       0.32      0.59      0.42      6008
                   check_mark       0.38      0.42      0.40      6000
            check_mark_button       0.37      0.43      0.40      6000
                   clown_face       0.28      0.31      0.29      6000
                      cooking       0.44      0.41      0.43      6000
                          egg       0.34      0.40      0.37      6000
                 enraged_face       0.21      0.25      0.23      6000
                         eyes       0.17      0.15      0.16      6000
      face_holding_back_tears       0.16      0.18      0.17      6000
           face_savoring_food       0.30      0.31      0.31      6000
    face_with_steam_from_nose       0.13      0.10      0.11      6000
       face_with_tears_of_joy       0.25      0.24    

(                                                     Text               label
 0       @PastorAlexLove Thank you, pastor. My mouth sh...  face_savoring_food
 1       So horny right now, sending pics of my thick h...  face_savoring_food
 2              😋  I will be quiet cause she already know.  face_savoring_food
 3       tonights supper is fake bake and chips. tomorr...  face_savoring_food
 4                              Bout to make my linguini 😋  face_savoring_food
 ...                                                   ...                 ...
 860023  @chrissennello @CodifyBaseball There’s not a s...          clown_face
 860024  Okay I can spoil how my friend felt about seei...          clown_face
 860025  @Hazel29630360 @FBM_Warrior @JoDivaRunner 🤡 yo...          clown_face
 860026  @KrumelsBunker @EmeraldRobinson @GeorgeMcfly17...          clown_face
 860027  @catturd2 Yes, good morning brainwashed😵‍💫 rig...          clown_face
 
 [860028 rows x 2 columns],
 817682         saluti

In [ ]:
#prevent svm from printing everytime we call the main

def svm_without_print():
    return main(repeat=False)

In [ ]:
# adding difficulty ranking analysis

def difficulty_analysis(y_true, y_predicted):
    dictionary_report = classification_report(y_true,y_predicted,output_dict=True)


    results = []
    non = ["accuracy", "weighted avg", "macro avg"]

    for i, j in dictionary_report.items():
        if i in non:
            continue
        results.append({"emoji": i, "f1": j["f1-score"], "precision": j["precision"],
                        "recall": j["recall"], "support": j["support"]})


    df = pd.DataFrame(results)
    df = df.sort_values("f1", ascending = False).reset_index(drop=True)
    return df


In [ ]:
def difficulty():

    df, y_test, y_predicted = svm_without_print()

    dataframe = difficulty_analysis(y_test, y_predicted)

    print("\nDiffuculty Ranking by Emoji\n", dataframe)

    print("\nEasiest 15 Emojis\n", dataframe.head(15))

    print("\nHardest 15 Emojis\n", dataframe.tail(15))


difficulty()

Classification by emoji:

                               precision    recall  f1-score   support

backhand_index_pointing_right       0.32      0.59      0.42      6008
                   check_mark       0.38      0.42      0.40      6000
            check_mark_button       0.37      0.43      0.40      6000
                   clown_face       0.28      0.31      0.29      6000
                      cooking       0.44      0.41      0.43      6000
                          egg       0.34      0.40      0.37      6000
                 enraged_face       0.21      0.25      0.23      6000
                         eyes       0.17      0.15      0.16      6000
      face_holding_back_tears       0.16      0.18      0.17      6000
           face_savoring_food       0.30      0.31      0.31      6000
    face_with_steam_from_nose       0.13      0.10      0.11      6000
       face_with_tears_of_joy       0.25      0.24      0.24      6000
                 fearful_face       0.17      0.24

In [ ]:
from sklearn.metrics import confusion_matrix


In [ ]:
#confusion matrices


def confusion():
    df, y_test, y_predicted = svm_without_print()
    uniques = sorted(df["label"].unique())

    conf = confusion_matrix(y_test, y_predicted, labels=uniques)

    df = pd.DataFrame(conf)

    df.index = uniques
    df.index.name = "Actual Emoji"

    df.columns = uniques
    df.columns.name = "Predicted Emoji"



#determining most confused emojis

    confused_emojis = []

    for i,actual in enumerate(uniques):
        for j,prediction in enumerate(uniques):
            if i==j:
                continue
                #using iloc to prevent pandas from thinking (0,1) is a col key
            n = df.iloc[i,j]
            if n>0:
                confused_emojis.append({"true": actual, "predicted":prediction, "number":int(n)})

    confused_ems = pd.DataFrame(confused_emojis)
    mistaken_emojis = confused_ems.sort_values("number", ascending=False).reset_index(drop=True)


    print("\nMost Confused 20 Emojis\n", mistaken_emojis.head(20))

In [ ]:
confusion()

Classification by emoji:

                               precision    recall  f1-score   support

backhand_index_pointing_right       0.32      0.59      0.42      6008
                   check_mark       0.38      0.42      0.40      6000
            check_mark_button       0.37      0.43      0.40      6000
                   clown_face       0.28      0.31      0.29      6000
                      cooking       0.44      0.41      0.43      6000
                          egg       0.34      0.40      0.37      6000
                 enraged_face       0.21      0.25      0.23      6000
                         eyes       0.18      0.15      0.16      6000
      face_holding_back_tears       0.16      0.18      0.17      6000
           face_savoring_food       0.30      0.31      0.31      6000
    face_with_steam_from_nose       0.13      0.10      0.11      6000
       face_with_tears_of_joy       0.25      0.24      0.24      6000
                 fearful_face       0.17      0.24